In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pickle
import glob

In [ ]:
df = pd.DataFrame(columns=['model', 'dataset', 'method', 'args', 'iter', 'acc@1', 'acc@5'])

paths = glob.glob('./output/*/*/*')
for path in sorted(paths):
    if not os.path.isdir(path):
        continue

    model, dataset, method = path.split('/')[-3:]
    args = ', '.join(method.split('_')[1:])
    method = method.split('_')[0]
    acc1 = []
    acc5 = []

    for iter in os.listdir(path):
        if not os.path.exists(os.path.join(path, iter, 'results.pkl')):
            continue
        
        results = pickle.load(open(os.path.join(path, iter, 'results.pkl'), 'rb'))

        acc1.append(results['accuracy1'])
        acc5.append(results['accuracy5'])
        
    if len(acc1) > 0:
        df = df.append({
            'model': model,
            'dataset': dataset,
            'method': method,
            'args': args,
            'iter': len(acc1),
            'acc@1': f'{np.mean(acc1):.2f}±{np.std(acc1):.2f}',
            'acc@5': f'{np.mean(acc5):.2f}±{np.std(acc5):.2f}'
        }, ignore_index=True)
        # print(path)
print(df)


             model  dataset    method  args iter       acc@1       acc@5
0   conv2next-tiny  food101  baseline          1  88.18±0.00  97.31±0.00
1   conv2next-tiny  food101        sn          1  88.30±0.00  97.26±0.00
2   conv2next-tiny  food101       snr          1  88.20±0.00  97.40±0.00
3   conv2next-tiny  food101      snvr          1  88.05±0.00  97.26±0.00
4   conv2next-tiny  food101        wn          1  85.39±0.00  96.72±0.00
5   conv2next-tiny    stl10  baseline          1  90.21±0.00  99.51±0.00
6   conv2next-tiny    stl10        wn          1  86.42±0.00  99.26±0.00
7       eswin-tiny  food101  baseline  5e-4    1  84.54±0.00  95.89±0.00
8       eswin-tiny    stl10  baseline          1  81.06±0.00  98.99±0.00
9         resnet34    stl10  baseline          1  86.88±0.00  99.10±0.00
10        resnet34    stl10        sn          1  87.69±0.00  99.00±0.00
11        resnet34    stl10        wn          1  86.31±0.00  99.09±0.00


In [47]:
import torch.nn as nn

m = nn.Conv2d(3,4,3,1,1)
print(m.weight[0][0])
nn.utils.weight_norm(m)
print(m.weight_g)
print(m.weight_v[0][0])

tensor([[-0.1693, -0.1656,  0.1537],
        [-0.1031, -0.0407,  0.1825],
        [ 0.0989,  0.0875,  0.0717]], grad_fn=<SelectBackward>)
Parameter containing:
tensor([[[[0.6117]]],


        [[[0.6419]]],


        [[[0.5807]]],


        [[[0.5936]]]], requires_grad=True)
tensor([[-0.1693, -0.1656,  0.1537],
        [-0.1031, -0.0407,  0.1825],
        [ 0.0989,  0.0875,  0.0717]], grad_fn=<SelectBackward>)


In [52]:
import math
import weightwatcher as ww
import torch
from torchvision import models

model_name = 'resnet34'
datasets = ['cifar10', 'cifar100', 'tiny-imagenet', 'stl10', 'food101']
for dataset in datasets:
    dataset_path = f'./output/{model_name}/{dataset}'
    if not os.path.exists(dataset_path):
        continue
    for path in sorted(os.listdir(dataset_path)):
        if not os.path.exists(os.path.join(dataset_path, path, '0', 'ckpt_best.pt')):
            continue
        num_classes = 10 if dataset in ['cifar10', 'stl10'] else 100 if dataset == 'cifar100' else 200 if dataset == 'tiny-imagenet' else 101
        model = models.resnet34(num_classes=num_classes)
        model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        model.maxpool = torch.nn.Identity()

        ckpt = torch.load(os.path.join(dataset_path, path, '0', 'ckpt_best.pt'))
        model.load_state_dict(ckpt['model'], strict=False)

        watcher = ww.WeightWatcher(model=model)
        details = watcher.analyze(plot=False, layers=[])
        
        print(dataset, path, f'{details["alpha"].mean():.2f}')
    print()


AttributeError: module 'torch.linalg' has no attribute 'svdvals'